<a href="https://colab.research.google.com/github/AKSHAYAVISWA/HexaTraining/blob/main/Weekly_Capstone_Projects/Supply_Chain_Management/Week3_Intro_Into_Pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, datediff, count

spark = SparkSession.builder \
    .appName("SupplyChainAnalysis") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [3]:
import pandas as pd

orders = pd.DataFrame({
    "order_id":[101,102,103,104,105,106,107,108],
    "supplier":["ABC","XYZ","ABC","PQR","XYZ","ABC","PQR","XYZ"],
    "order_date":[
        "2024-01-01",
        "2024-01-02",
        "2024-01-03",
        "2024-01-04",
        "2024-01-05",
        "2024-01-06",
        "2024-01-07",
        "2024-01-08"
    ],
    "delivery_date":[
        "2024-01-04",
        "2024-01-10",
        "2024-01-05",
        "2024-01-12",
        "2024-01-06",
        "2024-01-15",
        "2024-01-08",
        "2024-01-18"
    ]
})

orders.to_csv("orders.csv", index=False)

orders

,order_id,supplier,order_date,delivery_date
0,101,ABC,2024-01-01,2024-01-04
1,102,XYZ,2024-01-02,2024-01-10
2,103,ABC,2024-01-03,2024-01-05
3,104,PQR,2024-01-04,2024-01-12
4,105,XYZ,2024-01-05,2024-01-06
5,106,ABC,2024-01-06,2024-01-15
6,107,PQR,2024-01-07,2024-01-08
7,108,XYZ,2024-01-08,2024-01-18


In [4]:
orders_df = spark.read.csv(
    "orders.csv",
    header=True,
    inferSchema=True
)

orders_df.show()

+--------+--------+----------+-------------+
|order_id|supplier|order_date|delivery_date|
+--------+--------+----------+-------------+
|     101|     ABC|2024-01-01|   2024-01-04|
|     102|     XYZ|2024-01-02|   2024-01-10|
|     103|     ABC|2024-01-03|   2024-01-05|
|     104|     PQR|2024-01-04|   2024-01-12|
|     105|     XYZ|2024-01-05|   2024-01-06|
|     106|     ABC|2024-01-06|   2024-01-15|
|     107|     PQR|2024-01-07|   2024-01-08|
|     108|     XYZ|2024-01-08|   2024-01-18|
+--------+--------+----------+-------------+



In [5]:
orders_df = orders_df.withColumn(
    "delivery_days",
    datediff(col("delivery_date"), col("order_date"))
)

orders_df.show()

+--------+--------+----------+-------------+-------------+
|order_id|supplier|order_date|delivery_date|delivery_days|
+--------+--------+----------+-------------+-------------+
|     101|     ABC|2024-01-01|   2024-01-04|            3|
|     102|     XYZ|2024-01-02|   2024-01-10|            8|
|     103|     ABC|2024-01-03|   2024-01-05|            2|
|     104|     PQR|2024-01-04|   2024-01-12|            8|
|     105|     XYZ|2024-01-05|   2024-01-06|            1|
|     106|     ABC|2024-01-06|   2024-01-15|            9|
|     107|     PQR|2024-01-07|   2024-01-08|            1|
|     108|     XYZ|2024-01-08|   2024-01-18|           10|
+--------+--------+----------+-------------+-------------+



In [6]:
delayed_df = orders_df.filter(col("delivery_days") > 5)

delayed_df.show()

+--------+--------+----------+-------------+-------------+
|order_id|supplier|order_date|delivery_date|delivery_days|
+--------+--------+----------+-------------+-------------+
|     102|     XYZ|2024-01-02|   2024-01-10|            8|
|     104|     PQR|2024-01-04|   2024-01-12|            8|
|     106|     ABC|2024-01-06|   2024-01-15|            9|
|     108|     XYZ|2024-01-08|   2024-01-18|           10|
+--------+--------+----------+-------------+-------------+



In [7]:
result = delayed_df.groupBy("supplier") \
                   .agg(count("*").alias("Delayed_Orders"))

result.show()

+--------+--------------+
|supplier|Delayed_Orders|
+--------+--------------+
|     PQR|             1|
|     XYZ|             2|
|     ABC|             1|
+--------+--------------+



In [8]:
result.coalesce(1).write \
      .mode("overwrite") \
      .option("header", True) \
      .csv("output_csv")

print("CSV Saved Successfully")

CSV Saved Successfully


In [9]:
result.write \
      .mode("overwrite") \
      .parquet("output_parquet")

print("Parquet Saved Successfully")

Parquet Saved Successfully


In [10]:
import os

print("CSV Output Files:")
print(os.listdir("output_csv"))

print("\nParquet Output Files:")
print(os.listdir("output_parquet"))

CSV Output Files:
['.part-00000-22021916-94ba-4142-9a52-e2f38410cd09-c000.csv.crc', 'part-00000-22021916-94ba-4142-9a52-e2f38410cd09-c000.csv', '._SUCCESS.crc', '_SUCCESS']

Parquet Output Files:
['.part-00000-3fc94bc9-f1a5-45c9-b073-a02b629ca716-c000.snappy.parquet.crc', 'part-00000-3fc94bc9-f1a5-45c9-b073-a02b629ca716-c000.snappy.parquet', '._SUCCESS.crc', '_SUCCESS']
